## Build Local Schema

In [0]:
%python


from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType

mySchema = StructType([
    StructField("country", StringType()),
    StructField("citizen", StringType())
])

source_path = 'dbfs:/FileStore/streaming/'

## Read Streaming Data

In [0]:
%python

df = (
    spark.readStream
    .format("csv")
    .schema(mySchema)
    .option("header", "true")
    .load(f"{source_path}")
)

## Write Streaming data to table

In [0]:
( 
df.writeStream
.options(checkpointLocation=f"{source_path}/checkpoint")
.outputMode("append")
.format("delta")
.queryName("NaimishStream")
#.start(f"{source_path}/output")
.table("`my-unity-catlog`.demo.my_stream")
)                               

# df.printSchema() # to print the schema
# dbutils.fs.rm('dbfs:/FileStore/streaming/checkpoint',True)
# .outputMode("append")  >>> check the checkpoint and read only new files
# .outputMode("complete") >>>> used with aggregate streaming data

In [0]:
%sql
select * from `my-unity-catlog`.demo.my_stream

country,citizen
India,5
USA,10
China,5
India,5
Canada,10
Brazil,50
India,10
USA,5
China,10
India,10


In [0]:
%sql
describe history  `my-unity-catlog`.demo.my_stream

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2025-03-22T06:30:31Z,4369870837235887,naimish.databricks@hotmail.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 1fdb8dd0-e469-4b2b-8ec3-b03266fdd516, epochId -> 1, statsOnLoad -> false)",null,List(4115594743317488),0318-065910-u8vj6lt,1,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 6, numOutputBytes -> 956, numAddedFiles -> 1)",null,Databricks-Runtime/16.2.x-cpu-ml-scala2.12
1,2025-03-22T06:28:02Z,4369870837235887,naimish.databricks@hotmail.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 1fdb8dd0-e469-4b2b-8ec3-b03266fdd516, epochId -> 0, statsOnLoad -> false)",null,List(4115594743317488),0318-065910-u8vj6lt,0,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 6, numOutputBytes -> 955, numAddedFiles -> 1)",null,Databricks-Runtime/16.2.x-cpu-ml-scala2.12
0,2025-03-22T06:27:58Z,4369870837235887,naimish.databricks@hotmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(4115594743317488),0318-065910-u8vj6lt,null,WriteSerializable,true,Map(),null,Databricks-Runtime/16.2.x-cpu-ml-scala2.12


## Time Travel 

In [0]:
%sql
select * from `my-unity-catlog`.demo.my_stream version as of 1

country,citizen
India,10
USA,5
China,10
India,10
Canada,40
Brazil,10
